#### Analysis 1 

In [ ]:
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
fold = 0
exp_path = "./experiments/troubleshooting/20260227_103603"
fold_path = os.path.join(exp_path, "fold_" + str(fold))

In [ ]:
# Confusion matrix
cm = pd.read_csv(os.path.join(fold_path, "confusion_matrix.csv"), index_col=0)

lbls = ["CN", "MCI", "AD"]
cm.index = lbls
cm.columns = lbls

cmap = sn.light_palette('seagreen', as_cmap=True)
sn.heatmap(cm/cm.sum().sum(), annot=True, cmap=cmap)


In [ ]:
# Diagnosis distribution

sn.set_theme(palette='pastel')
samples = pd.read_csv(os.path.join(exp_path, "samples.csv"))
diag_dist = samples.groupby("label")["PET"].nunique()

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        count = int(round(pct * total / 100.0))
        return f'{pct:.0f}%\n(n={count})'
    return inner

plt.pie(
    diag_dist,
    labels=["CN", "MCI", "AD"],
    startangle=90,
    wedgeprops=dict(width=0.5),
    autopct=autopct_format(diag_dist),
    pctdistance=0.7
)
plt.show()

In [ ]:
# Train, validation and test splits

sn.set_theme(palette='pastel')

splits = [ 4327, 1050, 586]
lbls = ["Training", "Validation", "Test"]

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        count = int(round(pct * total / 100.0))
        return f'{pct:.0f}%\n(n={count})'
    return inner

plt.pie(
    splits,
    labels=lbls,
    startangle=90,
    wedgeprops=dict(width=0.5),
    autopct=autopct_format(diag_dist),
    pctdistance=0.7
)
plt.show()


In [ ]:
# Confidence distributions 

preds = pd.read_csv(os.path.join(fold_path, "predictions.csv"))

lbls = ["CN", "MCI", "AD"]

fig, axes = plt.subplots(3,3, figsize=(15,15))

for i in range(3):
    for j in range(3):

        axes[i,j].set_title(f"true {lbls[i]}, pred { lbls[j] }")
        axes[i,j].hist( 
            preds[ (preds["target"] == i) & (preds["pred"] == j) ]["confidence"], 
            bins=np.linspace(0.4,1,20),
            alpha=0.9, 
            cumulative=False)
        
        axes[i,j].set_ylim([0,50])



In [ ]:
import torch
import json 
import yaml

from pkg.utils.instantiate import instantiate
from pkg.training.criterion import build_criterion

In [ ]:
# Read config 
with open(os.path.join(exp_path, "config.yaml")) as f:
    cfg = yaml.safe_load(f)

print(cfg)

In [ ]:
dm = instantiate(cfg["datamodule"])
dm.setup()
dm.set_fold(fold)

In [ ]:
model = instantiate(cfg["model"])
criterion = build_criterion(cfg["criterion"], train_labels=dm.train_labels)

model.set_criterion(criterion)
best_state = torch.load( os.path.join(fold_path, "model.ckpt"), weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(best_state)

In [ ]:
from tqdm import tqdm

model.eval()

preds = []
targets = []
confs = []

for batch in tqdm(dm.test_dataloader()):

    out = model.test_batch(batch, 0)
    preds.append(out['preds'])
    targets.append(out['targets'])
    confs.append(out['confs'])

# Compute global confusion matrix and classification report
preds = torch.cat(preds).numpy()
targets = torch.cat(targets).numpy()
confs = torch.cat(confs).numpy()


### Analysis 2

In [3]:
from pkg.utils.report import experiment_report
from pkg.utils.reproducibility import repo_state
import os

In [ ]:
path1 = "./experiments/fdg_mri_demog_complete_only/20260319_043211"
path2 = "./experiments/pet_fdg_demog_baseline/20260318_103319"

experiment_report(path1, 5, title="FDG + MRI + Demographics")
experiment_report(path2, 5, title="FDG + Demographics")


In [4]:
def analyze_predictions(path): 

    # Read commit
    with open(os.path.join(path, "commit.txt")) as f:
        commit = f.read().strip()

    # Read patch 
    with open(os.path.join(path, "patch.diff"), "rb") as f:
        patch = f.read()

    return commit, patch


In [5]:
commit, patch = analyze_predictions("./experiments/amyloid_fdg_complete_only/20260306_100048/")
commit

'b5d4c89ff6440c0b44d9161cf70132ed1885692e'

In [34]:
import subprocess

def _git(*args, input=None, cwd=None, text=True):
    out = subprocess.run(
        ["git", *args],
        capture_output=True,
        check=True,
        input=input,
        cwd=cwd,
    ).stdout

    if text:
        return out.strip()
    return out

def get_repo_state():
    return {
        "commit": _git("rev-parse", "HEAD"),
        "patch": _git("diff", "HEAD", text=False),
        "branch": _git("rev-parse", "--abbrev-ref", "HEAD")
    }

In [35]:
original_state = get_repo_state()

In [36]:
_git("reset", "--hard")

b'HEAD is now at 44d1bec Adding code for experiment reproduction'

In [37]:
_git("checkout", commit)

b''

In [38]:
root = _git("rev-parse", "--show-toplevel")
_git("apply","-",input=patch, cwd=root)

b''

In [39]:
_git("reset", "--hard")

b'HEAD is now at b5d4c89 Modified PoE model to use prior as expert. Improved forward() efficiency. Added option to load cached multimodal samples. Added Test dataset and datamodule with size specification to run quick test experiments.'

In [40]:
_git("checkout", original_state["branch"])

b"Your branch is up to date with 'origin/refactor-code'."

In [41]:
root = _git("rev-parse", "--show-toplevel").decode("utf8").strip()
_git("apply","-",input=original_state["patch"], cwd=root)

b''